# 📝 과제 01: 온라인 쇼핑 상담 모델을 API로 서비스하기

## 만들 결과

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">교안 01의 서빙 과정</strong>을 온라인 쇼핑 상담 모델에 적용합니다. 터미널에서 서버를 실행하고, API로 모델 목록·답변·메모리 로그를 확인합니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">서버의 모델 목록 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model_table</code>과 질문 하나의 응답 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">response</code></li>
<li style="margin:3px 0;">생성 상한을 낮춘 응답 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">short_response</code>와 KV 캐시 정보 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cache_info</code></li>
<li style="margin:3px 0;">질문 8개와 답변을 나란히 담은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer_table</code></li>
</ul>
</div>

## 준비: 데이터와 제공 코드

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">환경:</strong> 서버는 부록 01에서 준비한 RunPod에서, 이 노트북은 내 PC에서 실행합니다. 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료/.env</code>에는 부록에서 준비한 공개 주소와 API 키가 있어야 합니다. 다른 모델 서버는 종료합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">자료:</strong> SFT 단원의 쇼핑 상담 모델과 평가 질문 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_eval.jsonl</code>을 그대로 사용합니다. 각 행의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">question</code>이 질문, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reference_answer</code>가 정답 답변입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">제공 코드:</strong> 경로·설정·<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">load_jsonl</code>·시스템 메시지·평가 질문 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">eval_rows</code>를 준비합니다. 서버는 아래 터미널 명령으로 실행합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO (번호)</code>에 해당하는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>를 채우고 문항의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[자가채점]</code> 셀을 실행합니다. 1번부터 순서대로 풉니다.</li>
</ul>
</div>

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from openai import OpenAI
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
material_dir = Path(".").resolve()
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)
day70_dir = material_dir.parent / "day70_SFT_지도파인튜닝"
SERVER_URL = os.environ["VLLM_SERVER_URL"].rstrip("/")  # RunPod 공개 HTTPS 주소
VLLM_API_KEY = os.environ["VLLM_API_KEY"]
MAX_TOKENS = 256     # 답변으로 생성할 최대 토큰 수
pd.set_option("display.max_colwidth", None)

In [ ]:
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(기존 실습과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
SYSTEM_PROMPT_COMMERCE = "너는 온라인 쇼핑 상담 챗봇이다. 전자상거래법 조문에 근거해 존댓말로 간결히 답한다."
eval_rows = load_jsonl(day70_dir / "data" / "과제" / "commerce_eval.jsonl")

## 1. 서버 실행과 API 연결

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 저장한 모델을 다른 프로그램이 호출할 수 있도록 실행하고, API에서 부를 모델 이름을 확인합니다.</p>
<p style="margin:6px 0;">아래 명령은 <strong style="font-size:1.13em;color:#1f3a5f;">터미널에 그대로 입력</strong>합니다. 서버를 실행한 터미널은 유지하고 노트북에서 클라이언트를 작성합니다.</p>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습자료 폴더에서 새 RunPod 터미널</strong>을 열고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day75_vLLM_추론엔진_서빙</code> 폴더로 이동합니다. 서버 명령은 이 폴더의 RunPod 전용 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv.lock</code> 환경을 사용합니다. 부록 01에서 자동 생성하여 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 저장한 API 키를 입력하고, 이 터미널에서 서버를 계속 실행합니다.</p>
</div>

```bash
cd day75_vLLM_추론엔진_서빙
mkdir -p output
read -s -p "서버 API 키 입력: " VLLM_API_KEY
export VLLM_API_KEY
```

**복사해서 실행 · RunPod 터미널**

```bash
uv run vllm serve muahseol/enkoa-qwen3.5-0.8b-commerce-sft \
  --served-model-name commerce-sft \
  --dtype bfloat16 \
  --max-model-len 4096 \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/commerce_server.log
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 요청을 보낼 수 있습니다. RunPod 터미널은 그대로 두고 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 노트북</strong>으로 돌아옵니다. 첫 실행은 모델 다운로드와 GPU 계산 준비에 시간이 걸릴 수 있습니다.</p>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">OpenAI</code>로 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">client</code></strong>를 만듭니다. 공개 주소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVER_URL</code> 뒤에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1</code>을 붙이고, API 키는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에서 읽은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>를 사용합니다.</li>
<li style="margin:3px 0;">(2) 서버의 모델 목록을 받아 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">model_list</code></strong>에 담습니다.</li>
<li style="margin:3px 0;">(3) 제공 코드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model_rows</code>를 테이블로 만들고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">id</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">root</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_model_len</code> 열을 이 순서로 골라 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">model_table</code></strong>에 담습니다.</li>
</ul>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 클라이언트 연결과 모델 목록을 참고합니다.
서버 주소는 base_url에 지정합니다. 모델 목록의 data에 있는 각 항목을 딕셔너리로 바꾼 행들이 제공되어 있습니다.
```

</details>

In [ ]:
# TODO (1): OpenAI 클라이언트에 vLLM 서버 주소를 지정합니다.
client = ...
# TODO (2): 서버의 모델 목록을 받습니다.
model_list = ...

model_rows = [model.to_dict() for model in model_list.data]
# TODO (3): 세 열을 가진 model_table을 만듭니다.
model_table = ...
display(model_table)

In [ ]:
# [자가채점] 연결 주소와 서버의 모델 목록을 확인합니다.
assert str(client.base_url).rstrip("/") == f"{SERVER_URL}/v1", "base_url에 서버 주소와 /v1을 지정하세요."
assert list(model_table.columns) == ["id", "root", "max_model_len"], "요구사항의 세 열을 순서대로 선택하세요."
assert list(model_table["id"]) == ["commerce-sft"], "터미널에서 쇼핑 상담 서버를 실행했는지 확인하세요."
assert int(model_table.iloc[0]["max_model_len"]) == 4096, "서버 실행 명령에 --max-model-len 4096을 지정했는지 확인하세요."
print("1번 완료")

## 2. 질문 하나 보내기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 모델 목록에서 확인한 이름으로 질문을 보내고, 응답에서 답변을 꺼냅니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) 시스템 메시지 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SYSTEM_PROMPT_COMMERCE</code>와 첫 질문 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">eval_rows[0][&quot;question&quot;]</code>을 담은 메시지 리스트 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">messages</code></strong>를 만듭니다.</li>
<li style="margin:3px 0;">(2) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">client.chat.completions.create</code>로 요청하고 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">response</code></strong>에 담습니다. 모델 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce-sft</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">temperature</code>는 0, 생성 상한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_completion_tokens</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">MAX_TOKENS</code>입니다.</li>
<li style="margin:3px 0;">(3) 첫 선택지의 답 문자열을 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">first_answer</code></strong>에 담습니다. 제공 코드가 답변·종료 이유·토큰 수를 표시합니다.</li>
</ul>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 질문 하나 보내기를 참고합니다.
메시지는 role과 content를 가진 딕셔너리 두 개입니다. 응답의 choices 첫 항목에서 message.content를 읽습니다.
```

</details>

In [ ]:
# TODO (1): 시스템 메시지와 사용자 질문을 구성합니다.
messages = ...
# TODO (2): 쇼핑 상담 서버에 그리디 생성 요청을 보냅니다.
response = ...
# TODO (3): 응답에서 답 문자열을 꺼냅니다.
first_answer = ...
print(first_answer)
response_info = {"종료 이유": response.choices[0].finish_reason,
                 "입력 토큰 수": response.usage.prompt_tokens,
                 "생성 토큰 수": response.usage.completion_tokens}
display(response_info)

In [ ]:
# [자가채점] 응답의 모델 이름·답변 추출·생성 상한을 확인합니다.
assert messages[0] == {"role":"system", "content":SYSTEM_PROMPT_COMMERCE}, "시스템 메시지를 확인하세요."
assert messages[1] == {"role":"user", "content":eval_rows[0]["question"]}, "첫 평가 질문을 넣으세요."
assert response.model == "commerce-sft", "요청의 model에는 서버의 모델 이름을 씁니다."
assert isinstance(first_answer, str) and first_answer.strip(), "응답의 답 문자열을 꺼내세요."
assert first_answer == response.choices[0].message.content, "첫 선택지의 답변과 일치해야 합니다."
assert 0 < response.usage.completion_tokens <= MAX_TOKENS, "요청의 생성 상한을 확인하세요."
print("2번 완료")

## 3. 생성 상한을 낮춰 요청하기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 같은 질문에서 생성 상한만 바꿔 답변과 종료 이유를 확인합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) 2번의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">messages</code>를 같은 서버에 그리디로 보내 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">short_response</code></strong>에 담습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_completion_tokens</code>만 8로 바꿉니다.</li>
</ul>
<p style="margin:6px 0;">종료 이유는 실제 응답을 읽습니다. 상한을 낮춰도 짧은 답변은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">stop</code>으로 끝날 수 있습니다.</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 생성 상한을 낮춰 비교하기를 참고합니다.
새 메시지를 만들지 않고 messages를 그대로 사용합니다. max_completion_tokens만 바꾸고 나머지 요청 조건은 유지합니다.
```

</details>

In [ ]:
# TODO (1): 생성 상한 8토큰으로 같은 질문을 보냅니다.
short_response = ...
print("답변:", short_response.choices[0].message.content)
print("종료 이유:", short_response.choices[0].finish_reason)

In [ ]:
# [자가채점] 답변 문구나 종료 이유를 고정하지 않고 생성 상한을 확인합니다.
assert short_response.model == "commerce-sft", "같은 쇼핑 상담 서버에 요청하세요."
assert 0 < short_response.usage.completion_tokens <= 8, "생성 상한을 8토큰으로 지정하세요."
assert isinstance(short_response.choices[0].message.content, str), "응답 객체를 short_response에 담으세요."
print("3번 완료")

## 4. 시작 로그 읽기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 서버가 확보한 KV 캐시 용량을 읽습니다. 동시 요청 수는 1번 모델 목록의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_model_len</code>을 기준으로 한 메모리 용량 추정치입니다.</p>
<p style="margin:6px 0;">RunPod의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/commerce_server.log</code>를 내려받아 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 단원 폴더 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">output</code></strong>에 같은 이름으로 저장합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_dir / &quot;commerce_server.log&quot;</code>를 UTF-8 문자열 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">log_text</code></strong>로 읽습니다.</li>
<li style="margin:3px 0;">(2) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU KV cache size: &lt;숫자&gt; tokens</code>의 숫자에서 쉼표를 지워 정수 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">kv_cache_tokens</code></strong>에 담습니다.</li>
<li style="margin:3px 0;">(3) 별도 줄의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Maximum concurrency for ...: &lt;숫자&gt;x</code>에서 x 앞 숫자를 실수 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">max_concurrency</code></strong>에 담습니다.</li>
</ul>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 시작 로그의 메모리 용량을 참고합니다.
read_text로 전체 로그를 읽습니다. re.search에서 괄호로 묶은 숫자를 group(1)로 꺼내 각각 int와 float로 바꿉니다.
```

</details>

In [ ]:
# TODO (1): 같은 서버의 시작 로그를 읽습니다.
log_text = ...
for line in log_text.splitlines():
    if "GPU KV cache size" in line or "Maximum concurrency" in line:
        print(line)
# TODO (2): KV 캐시 토큰 수를 정수로 꺼냅니다.
kv_match = ...
kv_cache_tokens = ...
# TODO (3): 동시 요청 추정치를 실수로 꺼냅니다.
concurrency_match = ...
max_concurrency = ...
cache_info = {"KV 캐시(토큰)":kv_cache_tokens, "동시 요청 추정치":max_concurrency}
display(cache_info)

In [ ]:
# [자가채점] 자료형과 실제 시작 로그의 값을 확인합니다.
assert isinstance(kv_cache_tokens, int) and kv_cache_tokens > 0, "토큰 수는 양의 정수입니다."
assert isinstance(max_concurrency, float) and max_concurrency > 0, "동시 요청 추정치는 양의 실수입니다."
assert f"GPU KV cache size: {kv_cache_tokens:,} tokens" in log_text, "KV 캐시 줄의 숫자를 확인하세요."
assert f"Maximum concurrency for {int(model_table.iloc[0]['max_model_len']):,} tokens per request: {max_concurrency:.2f}x" in log_text, "모델 목록의 최대 길이와 같은 서버 로그의 동시 요청 줄을 확인하세요."
print("4번 완료")

## 5. 여러 질문을 함께 보내기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 같은 서버를 LangChain 클라이언트로 연결해 여러 질문의 답을 받습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) 서버 이름 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce-sft</code>를 사용하는 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">commerce_llm</code></strong>을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatOpenAI</code>로 만듭니다. 주소·키는 1번과 같고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">temperature=0</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_tokens=MAX_TOKENS</code>입니다.</li>
<li style="margin:3px 0;">(2) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">eval_rows[:8]</code>의 질문마다 시스템 메시지와 사용자 질문의 튜플 리스트를 만들어 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">eval_messages</code></strong>에 순서대로 더합니다.</li>
<li style="margin:3px 0;">(3) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">batch</code>로 질문 묶음을 보내 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">concurrent_answers</code></strong>에 담습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_concurrency</code>는 4입니다. 제공 코드가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer_table</code>을 표시합니다.</li>
</ul>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01의 여러 질문을 한꺼번에 보내기를 참고합니다.
LangChain 메시지는 (역할, 내용) 튜플 리스트입니다. 역할은 system과 human을 사용하며, batch의 config에 동시 요청 상한을 지정합니다.
```

</details>

In [ ]:
# TODO (1): LangChain 클라이언트를 같은 서버에 연결합니다.
commerce_llm = ...
# TODO (2): 앞 8개 질문의 메시지를 순서대로 구성합니다.
eval_messages = []
for row in eval_rows[:8]:
    eval_messages.append(...)
# TODO (3): 동시에 진행할 요청 수를 4로 제한해 보냅니다.
concurrent_answers = ...
answer_table = pd.DataFrame({
    "질문":[row["question"] for row in eval_rows[:8]],
    "답변":[answer.content for answer in concurrent_answers],
})
display(answer_table)

In [ ]:
# [자가채점] 모든 질문의 응답과 결과 테이블을 확인합니다.
assert commerce_llm.model_name == "commerce-sft", "쇼핑 상담 모델 이름을 지정하세요."
expected_messages = [[("system", SYSTEM_PROMPT_COMMERCE), ("human", row["question"])] for row in eval_rows[:8]]
assert eval_messages == expected_messages, "평가 질문 앞 8개와 시스템 메시지가 순서대로 대응해야 합니다."
assert len(eval_messages) == len(concurrent_answers) == 8, "평가 질문 앞 8개를 순서대로 보내세요."
assert list(answer_table.columns) == ["질문","답변"], "제공 코드의 결과 열을 유지하세요."
assert all(isinstance(answer.content,str) and answer.content.strip() for answer in concurrent_answers), "비어 있는 응답을 확인하세요."
print("5번 완료")

### 마무리: 서버 종료

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod에서 서버를 실행한 터미널</strong>로 돌아가 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code></strong>를 누릅니다. 프롬프트가 돌아오면 같은 터미널에서 확인합니다.</p>
</div>

```bash
nvidia-smi
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 서버 프로세스가 사라지고 GPU 메모리 사용량이 실행 전 수준으로 줄었는지</strong> 확인합니다. 서버 프로세스가 종료되면 모델 가중치와 KV 캐시 메모리도 반환됩니다. 확인을 마친 뒤 다음 단계로 이동합니다.</p>
</div>